In [1]:
!pip install pyspark

In [2]:
# Импорт библиотек
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, stddev, hour, to_timestamp,
    regexp_extract, when, lit, window, desc, asc,
    sum as spark_sum,round as spark_round, rand, randn, least,
    date_format
)

from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType, DoubleType
)

In [3]:
#Инициализация SparkSession
# Создание SparkSession
spark = SparkSession.builder \
    .appName("brooklyn_sales_map") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Установка уровня логирования
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

26/07/02 00:57:58 WARN Utils: Your hostname, devopsvm resolves to a loopback address: 127.0.1.1; using 192.168.0.137 instead (on interface enp0s3)
26/07/02 00:57:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 00:57:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.3
Spark UI: http://localhost:4040


In [4]:
# Загрузка данных из HDFS в Spark DataFrame
hdfs_path = "hdfs://localhost:9000/user/hadoop/task7/input/brooklyn_sales_map.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(hdfs_path)

print("Схема данных brooklyn_sales_map:")
df.printSchema()

print("Общая статистика brooklyn_sales_map:")
print(f"Всего записей: {df.count():,}")

print("\nПримеры данных:")
df.show(5, truncate=False)

Схема данных brooklyn_sales_map:
root
 |-- _c0: integer (nullable = true)
 |-- borough1: integer (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- building_class_category: string (nullable = true)
 |-- tax_class: string (nullable = true)
 |-- block: integer (nullable = true)
 |-- lot: integer (nullable = true)
 |-- easement: string (nullable = true)
 |-- building_class: string (nullable = true)
 |-- address9: string (nullable = true)
 |-- apartment_number: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- residential_units: integer (nullable = true)
 |-- commercial_units: integer (nullable = true)
 |-- total_units: integer (nullable = true)
 |-- land_sqft: double (nullable = true)
 |-- gross_sqft: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- tax_class_at_sale: integer (nullable = true)
 |-- building_class_at_sale: string (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |

Всего записей: 390,883

Примеры данных:


26/07/02 00:58:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+--------+---------------------+-----------------------+---------+-----+----+--------+--------------+-------------------+----------------+--------+-----------------+----------------+-----------+---------+----------+----------+-----------------+----------------------+------------+----------+------------+---------+---+------+------+----------+-------+-------+--------+----------+----------+----------+---------+----------+--------+-------------------+---------+---------+---------+---------+--------+--------+-------+-------+-------+---------+---------+---------+-------+---------+---------+---------------------+-------+--------+-------+-------+----------+----------+----------+---------+----------+---------+----------+--------+---------+--------+----------+--------+--------+---------+---------+---+--------+----------+-------+--------+----------+---------+----------+---------+---------+----------+----------+----------------------------------+--------+--------+--------+-------+--------+----

26/07/02 00:58:33 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

In [5]:
#ТРАНСФОРМАЦИЯ
#Фильтрация данных (цена или категория не указаны или маленькая цена)
filtered_df = df.filter(
    col("building_class_category").isNotNull() & 
    col("sale_price").isNotNull() & 
    (col("sale_price") > 10000)
)

#Создание колонки цена за кв фут
df_transformed = filtered_df.withColumn(
    "price_per_sqft",
    when(col("gross_sqft") > 0, col("sale_price") / col("gross_sqft")).otherwise(lit(None))
)

In [6]:
#Агрегация (группировка по районам и категориям зданий)
df_analytics = df_transformed \
    .groupBy("borough1", "building_class_category") \
    .agg(
        count("sale_price").alias("total_sales_count"),
        spark_round(avg("sale_price"), 2).alias("avg_sale_price"),
        spark_round(avg("price_per_sqft"), 2).alias("avg_price_per_sqft")
    ) \
    .orderBy(desc("total_sales_count")) # Сортируем по популярности направлений


In [7]:
# Действие
#Агрегированные результаты
print("Результаты трансформаций (аналитика по районам и категориям зданий):")
df_analytics.show(50, truncate=False)

Результаты трансформаций (аналитика по районам и категориям зданий):


26/07/02 00:59:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: borough, building_class_category, gross_sqft, sale_price
 Schema: borough1, building_class_category, gross_sqft, sale_price
Expected: borough1 but found: borough
CSV file: hdfs://localhost:9000/user/hadoop/task7/input/brooklyn_sales_map.csv


+--------+-------------------------------------------+-----------------+--------------+------------------+
|borough1|building_class_category                    |total_sales_count|avg_sale_price|avg_price_per_sqft|
+--------+-------------------------------------------+-----------------+--------------+------------------+
|3       |02 TWO FAMILY HOMES                        |62057            |645222.18     |390.51            |
|3       |01 ONE FAMILY HOMES                        |33853            |626988.53     |427.94            |
|3       |10  COOPS - ELEVATOR APARTMENTS            |28879            |310156.53     |11.31             |
|3       |13  CONDOS - ELEVATOR APARTMENTS           |25286            |697367.98     |402.77            |
|3       |03 THREE FAMILY HOMES                      |21347            |723969.15     |420.76            |
|3       |07  RENTALS - WALKUP APARTMENTS            |15330            |1213473.23    |1270.22           |
|3       |09  COOPS - WALKUP APARTMEN

In [8]:
#Вывод результатов
output_hdfs_path = "hdfs://localhost:9000/user/hadoop/task7/output/brooklyn_sales_map.csv"

print(f"Сохранение результатов в HDFS по пути: {output_hdfs_path}...")
df_analytics.write \
    .mode("overwrite") \
    .parquet(output_hdfs_path)

print("Результаты сохранены в HDFS")

#Сохранение результатов локально:
df_analytics.toPandas().to_csv('/tmp/df_analytics.csv', index=False)

Сохранение результатов в HDFS по пути: hdfs://localhost:9000/user/hadoop/task7/output/brooklyn_sales_map.csv...


26/07/02 00:59:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: borough, building_class_category, gross_sqft, sale_price
 Schema: borough1, building_class_category, gross_sqft, sale_price
Expected: borough1 but found: borough
CSV file: hdfs://localhost:9000/user/hadoop/task7/input/brooklyn_sales_map.csv


Результаты сохранены в HDFS


26/07/02 00:59:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: borough, building_class_category, gross_sqft, sale_price
 Schema: borough1, building_class_category, gross_sqft, sale_price
Expected: borough1 but found: borough
CSV file: hdfs://localhost:9000/user/hadoop/task7/input/brooklyn_sales_map.csv


In [9]:
# Остановка SparkSession
spark.stop()
print("SparkSession остановлен")

SparkSession остановлен
